# Лабораторная работа № 2. Индексация, фильтрация и поиск аномалий

**Кейс:** поиск финансовых аномалий в 12 заказах.
**Ограничения:** в обязательных заданиях нет циклов `for`/`while`, нет Pandas, нет ручного перебора. Используются векторизованные операции NumPy, булевы маски и агрегации. Все массивы связаны по позиции (элемент `i` описывает один заказ), поэтому везде применяются **согласованные маски**.

In [1]:
import numpy as np

order_id = np.arange(2001, 2013)

region = np.array([
    "Bishkek", "Osh", "Bishkek", "Karakol",
    "Osh", "Tokmok", "Bishkek", "Osh",
    "Karakol", "Tokmok", "Bishkek", "Osh"
])

quantity = np.array([2, 1, 5, 3, 2, 8, 1, 4, 2, 3, 6, 1])

unit_price = np.array([
    1500, 20000, 800, 5000, 1200, 700,
    25000, 3000, 4500, 2000, 900, 18000
])

discount = np.array([
    0.05, 0.10, 0.00, 0.15, 0.05, 0.20,
    0.30, 0.10, 0.00, 0.05, 0.50, 0.10
])

unit_cost = np.array([
    1000, 16000, 600, 4000, 900, 500,
    18000, 2200, 3500, 1500, 800, 14000
])

revenue = quantity * unit_price * (1 - discount)
profit = revenue - quantity * unit_cost
margin = profit / revenue      # рентабельность: доля прибыли в выручке

def ids(mask):
    "Номера заказов, попавших в маску."
    return order_id[mask]

# проверка согласованности исходных данных
assert order_id.shape == region.shape == quantity.shape == unit_price.shape == discount.shape == unit_cost.shape
print("Заказов:", order_id.size)
print("Выручка:", revenue)
print("Прибыль:", profit)

Заказов: 12
Выручка: [ 2850. 18000.  4000. 12750.  2280.  4480. 17500. 10800.  9000.  5700.
  2700. 16200.]
Прибыль: [  850.  2000.  1000.   750.   480.   480.  -500.  2000.  2000.  1200.
 -2100.  2200.]


## Задание 1. Первичный отбор

In [2]:
REV_THRESHOLD = 5000
REGION = "Osh"

mask_revenue = revenue > REV_THRESHOLD
mask_qty = quantity >= 3
mask_region = region == REGION

print("revenue > 5000 :", mask_revenue.astype(int), "->", ids(mask_revenue), "| выручка", revenue[mask_revenue])
print("quantity >= 3  :", mask_qty.astype(int), "->", ids(mask_qty), "| кол-во", quantity[mask_qty])
print(f"region == {REGION}:", mask_region.astype(int), "->", ids(mask_region))
print("\nЧисло заказов по условию:", mask_revenue.sum(), mask_qty.sum(), mask_region.sum())

revenue > 5000 : [0 1 0 1 0 0 1 1 1 1 0 1] -> [2002 2004 2007 2008 2009 2010 2012] | выручка [18000. 12750. 17500. 10800.  9000.  5700. 16200.]
quantity >= 3  : [0 0 1 1 0 1 0 1 0 1 1 0] -> [2003 2004 2006 2008 2010 2011] | кол-во [5 3 8 4 3 6]
region == Osh: [0 1 0 0 1 0 0 1 0 0 0 1] -> [2002 2005 2008 2012]

Число заказов по условию: 7 6 4


**Вывод.** Каждое условие даёт собственную булеву маску длины 12, а отбор `array[mask]` одинаково работает для любого связанного массива (`order_id`, `revenue`, `quantity`).
- Порог 5 000 пропускает 7 из 12 заказов и отсекает мелкие чеки (2001, 2003, 2005, 2006, 2011).
- Условие `quantity >= 3` выбирает другой набор: много единиц не означает большую выручку (2003: 5 шт., но выручка всего 4 000, так как цена 800). Количество и выручка - разные измерения.
- Регион Osh - 4 заказа (2002, 2005, 2008, 2012).

## Задание 2. Составные условия

In [3]:
high_disc = discount >= 0.20
big_rev = revenue > 10_000
big_qty = quantity >= 4

print("Отдельные условия:")
print("  скидка >= 20%      :", ids(high_disc))
print("  выручка > 10 000   :", ids(big_rev))
print("  количество >= 4    :", ids(big_qty))
print("\nПары:")
print("  скидка >= 20% & выручка > 10 000   :", ids(high_disc & big_rev))
print("  скидка >= 20% & количество >= 4    :", ids(high_disc & big_qty))
print("  выручка > 10 000 & количество >= 4 :", ids(big_rev & big_qty))

chosen = high_disc & big_rev
print("\nВыбранная комбинация -> заказы", ids(chosen))
print("выручка :", revenue[chosen], "прибыль:", profit[chosen], "рентабельность:", margin[chosen].round(3))

Отдельные условия:
  скидка >= 20%      : [2006 2007 2011]
  выручка > 10 000   : [2002 2004 2007 2008 2012]
  количество >= 4    : [2003 2006 2008 2011]

Пары:
  скидка >= 20% & выручка > 10 000   : [2007]
  скидка >= 20% & количество >= 4    : [2006 2011]
  выручка > 10 000 & количество >= 4 : [2008]

Выбранная комбинация -> заказы [2007]
выручка : [17500.] прибыль: [-500.] рентабельность: [-0.029]


**Выбор комбинации: «скидка >= 20 % И выручка > 10 000».** Цель анализа - найти места, где компания теряет деньги: большая скидка важна там, где под ней крупная сумма сделки. Эта пара находит заказ **2007** (скидка 30 %, выручка 17 500), и он оказывается убыточным (прибыль -500).

Другие комбинации менее информативны:
- «скидка >= 20 % и количество >= 4» даёт 2006 и 2011: оптовая покупка с крупной скидкой - обычная практика, суммы небольшие (4 480 и 2 700), финансовый риск мал;
- «выручка > 10 000 и количество >= 4» оставляет только 2008 - здоровый заказ без скидочного риска.

Оговорка: выбранное правило **пропускает** убыточный 2011 (выручка 2 700 < 10 000). Пороги - аналитическое решение, и один найденный заказ - повод для проверки, а не доказательство качества правила.

## Задание 3. Поиск убыточных заказов

In [4]:
loss_mask = profit < 0
loss_share = loss_mask.mean()

print("Убыточные заказы :", ids(loss_mask))
print("Выручка          :", revenue[loss_mask])
print("Прибыль          :", profit[loss_mask])
print("Скидка           :", discount[loss_mask])
print(f"Доля убыточных   : {loss_mask.sum()} из {loss_mask.size} = {loss_share:.1%}")
print(f"Суммарный убыток : {profit[loss_mask].sum():,.0f}  | прибыль по остальным: {profit[~loss_mask].sum():,.0f}")

# ранги: 1 = наибольшее значение
rank_rev = (-revenue).argsort().argsort() + 1
rank_prof = (-profit).argsort().argsort() + 1
top5 = (-revenue).argsort()[:5]
print("\nТоп-5 по выручке: заказ, выручка, прибыль, ранг по выручке, ранг по прибыли")
print(np.column_stack((order_id[top5], revenue[top5], profit[top5], rank_rev[top5], rank_prof[top5])))
print("\nКорреляция выручки и прибыли:", np.corrcoef(revenue, profit)[0, 1].round(3))

Убыточные заказы : [2007 2011]
Выручка          : [17500.  2700.]
Прибыль          : [ -500. -2100.]
Скидка           : [0.3 0.5]
Доля убыточных   : 2 из 12 = 16.7%
Суммарный убыток : -2,600  | прибыль по остальным: 12,960

Топ-5 по выручке: заказ, выручка, прибыль, ранг по выручке, ранг по прибыли
[[ 2.002e+03  1.800e+04  2.000e+03  1.000e+00  2.000e+00]
 [ 2.007e+03  1.750e+04 -5.000e+02  2.000e+00  1.100e+01]
 [ 2.012e+03  1.620e+04  2.200e+03  3.000e+00  1.000e+00]
 [ 2.004e+03  1.275e+04  7.500e+02  4.000e+00  8.000e+00]
 [ 2.008e+03  1.080e+04  2.000e+03  5.000e+00  4.000e+00]]

Корреляция выручки и прибыли: 0.38


**Вывод.** Убыточны 2 заказа из 12 (**16,7 %**): **2007** и **2011**. Суммарный убыток 2 600; прибыль по остальным 10 заказам 12 960 (портфель в плюсе, но убытки уменьшают общую прибыль примерно на пятую часть).

**Почему высокая выручка не гарантирует прибыль.** Прибыль = выручка - себестоимость; выручка зависит от цены, скидки и количества, но не учитывает затраты. Заказ 2007 - второй по выручке (17 500), а по прибыли только 11-й из 12: цена со скидкой `25 000 * 0,7 = 17 500` ниже себестоимости `18 000`. Заказ 2002 - первый по выручке (18 000), но по прибыли лишь второй (2 000 при себестоимости 16 000). Корреляция выручки и прибыли всего около 0,38, то есть выручка - слабый ориентир. Для оценки заказа нужна рентабельность `profit / revenue`, а не одна выручка.

## Задание 4. Анализ скидок

In [5]:
low = discount <= 0.05
high = discount >= 0.20
mid = ~low & ~high

def summary(name, m):
    print(f"{name:14s} n={m.sum()}  заказы={ids(m)}  ср.выручка={revenue[m].mean():9.1f}  "
          f"ср.прибыль={profit[m].mean():8.1f}  ср.рентаб.={margin[m].mean():6.1%}  убыточных={loss_mask[m].sum()}")

summary("низкая <=5%", low)
summary("средняя", mid)
summary("высокая >=20%", high)

risky = high & (margin < 0.05)
print("\nСкидка >= 20% и рентабельность < 5% (в т.ч. убыток):", ids(risky))
print("рентабельность:", margin[risky].round(3), "| скидка:", discount[risky])

# предельная безубыточная скидка: price*(1-d) = cost  ->  d* = 1 - cost/price
break_even = 1 - unit_cost / unit_price
over = discount > break_even
print("\nЗаказ, скидка, безубыточная скидка (только там, где скидка выше):")
print(np.column_stack((order_id, discount, break_even.round(3)))[over])

низкая <=5%    n=5  заказы=[2001 2003 2005 2009 2010]  ср.выручка=   4766.0  ср.прибыль=  1106.0  ср.рентаб.= 23.8%  убыточных=0
средняя        n=4  заказы=[2002 2004 2008 2012]  ср.выручка=  14437.5  ср.прибыль=  1737.5  ср.рентаб.= 12.3%  убыточных=0
высокая >=20%  n=3  заказы=[2006 2007 2011]  ср.выручка=   8226.7  ср.прибыль=  -706.7  ср.рентаб.=-23.3%  убыточных=2

Скидка >= 20% и рентабельность < 5% (в т.ч. убыток): [2007 2011]
рентабельность: [-0.029 -0.778] | скидка: [0.3 0.5]

Заказ, скидка, безубыточная скидка (только там, где скидка выше):
[[2.007e+03 3.000e-01 2.800e-01]
 [2.011e+03 5.000e-01 1.110e-01]]


**Вывод.** Группа с высокой скидкой (>= 20 %: 2006, 2007, 2011) показывает отрицательную среднюю рентабельность, и 2 из 3 заказов убыточны. В группе низкой скидки (<= 5 %: 2001, 2003, 2005, 2009, 2010) убыточных нет. Заказы, сочетающие большую скидку и плохой результат, - **2007** и **2011**.

Причинный вывод «скидка вызывает убыток» делать **нельзя**:
- групп мало (3 и 5 наблюдений), а один заказ (2011, рентабельность -78 %) определяет среднее;
- скидки назначались не случайно: возможно, крупные скидки давали на товары с низкой наценкой или при распродаже;
- 2006 имеет скидку 20 %, но остаётся прибыльным, так как запас между ценой и себестоимостью достаточен.

Арифметически убыток возникает, когда скидка превышает безубыточную `1 - cost/price`: для 2007 это около 28 % против фактических 30 %, для 2011 около 11 % против 50 %. Это описывает механизм, но не объясняет, почему скидка была предоставлена.

## Задание 5. Выявление подозрительных наблюдений

In [6]:
# Правило A: очень высокая скидка при низкой рентабельности
rule_a = (discount >= 0.30) & (margin < 0.05)

# Правило B: необычно высокая цена (выше Q3 + 1.5*IQR) при малом количестве
q1, q3 = np.percentile(unit_price, [25, 75])
price_fence = q3 + 1.5 * (q3 - q1)
print(f"Q1={q1:.0f}, Q3={q3:.0f}, верхняя граница={price_fence:.0f}, медиана={np.median(unit_price):.0f}")
rule_b = (unit_price > price_fence) & (quantity <= 1)

# Правило C: скидка выше безубыточной
rule_c = discount > (1 - unit_cost / unit_price)

print("A скидка>=30% & рентаб.<5% :", ids(rule_a))
print("B дорогая цена & кол-во<=1 :", ids(rule_b))
print("C скидка > безубыточной    :", ids(rule_c))

suspicious = rule_a | rule_b | rule_c
print("\nВсе подозрительные:", ids(suspicious))

needs_check = rule_a | rule_c
explained = rule_b & ~needs_check & (margin > 0.10)
print("Требуют проверки   :", ids(needs_check))
print("Вероятно объяснимы :", ids(explained), "| рентабельность:", margin[explained].round(3))

Q1=1125, Q3=8250, верхняя граница=18938, медиана=2500
A скидка>=30% & рентаб.<5% : [2007 2011]
B дорогая цена & кол-во<=1 : [2002 2007]
C скидка > безубыточной    : [2007 2011]

Все подозрительные: [2002 2007 2011]
Требуют проверки   : [2007 2011]
Вероятно объяснимы : [2002] | рентабельность: [0.111]


**Вывод.**
- **Требуют дополнительной проверки: 2007 и 2011.** Продажа ниже себестоимости при скидке 30 % и 50 % (правила A и C срабатывают на обоих). Нужно выяснить, кто утвердил скидку, не ошибка ли ввода (например, 50 % вместо 5 %), не акция ли это.
- **Объясним обычной ситуацией: 2002.** Цена 20 000 при количестве 1 выглядит выбросом (медиана цен 2 500, граница 18 938), но, вероятно, это дорогой штучный товар: скидка 10 %, рентабельность 11 % нормальна. Заказ 2012 (цена 18 000, количество 1) лишь немного не дотягивает до границы правила B и по смыслу такой же. Высокая цена 2007 тоже срабатывает по правилу B, но этот заказ уже в проверке из-за убытка.
- Заказ 2006 (скидка 20 %, количество 8, прибыль 480) правила не затронули: типичная оптовая скидка с положительной прибылью, можно лишь наблюдать.

Цена сама по себе не ошибка: она тревожна только в сочетании со скидкой и себестоимостью.

## Задание 6. Анализ сегментов

In [7]:
m_bishkek = region == "Bishkek"
m_osh = region == "Osh"
total_rev = revenue.sum()

print("Регион  | заказов | выручка | доля выручки | прибыль | убыточных | ср.скидка")
print("Bishkek |", m_bishkek.sum(), "|", revenue[m_bishkek].sum(), "|", round(revenue[m_bishkek].sum() / total_rev, 3),
      "|", profit[m_bishkek].sum(), "|", loss_mask[m_bishkek].sum(), "|", discount[m_bishkek].mean().round(3))
print("Osh     |", m_osh.sum(), "|", revenue[m_osh].sum(), "|", round(revenue[m_osh].sum() / total_rev, 3),
      "|", profit[m_osh].sum(), "|", loss_mask[m_osh].sum(), "|", discount[m_osh].mean().round(3))

seg_mask = m_bishkek | m_osh
print("\nОбщая маска сегмента:", seg_mask.sum(), "заказов; id:", ids(seg_mask))
print("Согласованность длин:", order_id[seg_mask].shape == revenue[seg_mask].shape == profit[seg_mask].shape == region[seg_mask].shape)
print("Убыточные в Bishkek:", ids(m_bishkek & loss_mask), "| в Osh:", ids(m_osh & loss_mask))

Регион  | заказов | выручка | доля выручки | прибыль | убыточных | ср.скидка
Bishkek | 4 | 27050.0 | 0.255 | -750.0 | 2 | 0.212
Osh     | 4 | 47280.0 | 0.445 | 6680.0 | 0 | 0.088

Общая маска сегмента: 8 заказов; id: [2001 2002 2003 2005 2007 2008 2011 2012]
Согласованность длин: True
Убыточные в Bishkek: [2007 2011] | в Osh: []


**Вывод.** Оба региона имеют по 4 заказа, но результат разный:

| Показатель | Bishkek | Osh |
|---|---:|---:|
| Заказов | 4 | 4 |
| Выручка | 27 050 | 47 280 |
| Прибыль | -750 | 6 680 |
| Убыточных заказов | 2 (2007, 2011) | 0 |

Osh приносит на 75 % больше выручки при том же числе заказов и без убытков; оба убыточных заказа в Bishkek. Но по 4 наблюдения на регион: разницу создают два заказа с экстремальной скидкой, и это не обязательно особенность региона. Одинаковые маски применялись ко всем связанным массивам, поэтому строки остались согласованными.

## Задание 7. Влияние аномалий

In [8]:
base_mean = revenue.mean()
base_med = np.median(revenue)
print(f"До исключения: среднее={base_mean:.1f}, медиана={base_med:.1f}, n={revenue.size}")

rq1, rq3 = np.percentile(revenue, [25, 75])
rev_outlier = revenue > rq3 + 1.5 * (rq3 - rq1)

scenarios = {
    "A: скидка>=30% & рентаб.<5%": rule_a,
    "B: дорогая цена & кол-во<=1": rule_b,
    "A|B|C (все правила)": suspicious,
    "выручка > Q3+1.5*IQR": rev_outlier,
}
print("\nСценарий | исключено | среднее после | изменение | медиана после")
for name, m in scenarios.items():
    keep = ~m
    print(f"{name:30s} {str(ids(m)):26s} {revenue[keep].mean():9.1f} {revenue[keep].mean() / base_mean - 1:+8.1%} {np.median(revenue[keep]):9.1f}")

До исключения: среднее=8855.0, медиана=7350.0, n=12

Сценарий | исключено | среднее после | изменение | медиана после
A: скидка>=30% & рентаб.<5%    [2007 2011]                   8606.0    -2.8%    7350.0
B: дорогая цена & кол-во<=1    [2002 2007]                   7076.0   -20.1%    5090.0
A|B|C (все правила)            [2002 2007 2011]              7562.2   -14.6%    5700.0
выручка > Q3+1.5*IQR           []                            8855.0    +0.0%    7350.0


**Вывод.** Результат заметно зависит от правила (точные значения см. в таблице выше):
- средняя выручка по всем 12 заказам 8 855, медиана 7 350;
- правило A (исключены 2007, 2011) снижает среднее лишь на 2,8 %: один исключённый заказ крупный (17 500), другой мелкий (2 700), эффекты частично гасятся;
- правило B (2002, 2007) удаляет два самых дорогих штучных заказа, и среднее падает на 20,1 % (до 7 076), медиана - до 5 090;
- объединение всех правил даёт -14,6 %;
- статистическое правило `Q3 + 1.5 IQR` по выручке выбросов не находит, среднее остаётся прежним.

Медиана меняется заметно меньше среднего, то есть устойчивее к выбору правила. Поэтому нельзя автоматически убирать большие значения: 2002 и 2012 - обычные прибыльные дорогие товары, их исключение занизило бы реальную выручку. Правильнее показывать оба варианта (с аномалиями и без) и удалять наблюдения только после проверки; убыточные 2007 и 2011 - кандидаты на проверку, а не на удаление.

## Задание 8. Аналитический отчёт

**1. Обнаруженные аномалии**
- Заказы **2007** (скидка 30 %, прибыль -500) и **2011** (скидка 50 %, прибыль -2 100) проданы ниже себестоимости: скидка превышает безубыточную (около 28 % и 11 %).
- Заказы **2002** (по правилу цены) и **2012** (чуть ниже границы) - необычно дорогие позиции при количестве 1; по прибыли выглядят нормально.
- Заказ **2006** - скидка 20 % при количестве 8; прибыльный, но с невысокой рентабельностью.

**2. Что влияет на финансовые выводы**
- 2 убыточных заказа (16,7 %) дают убыток 2 600 при прибыли 12 960 по остальным заказам.
- 2007 и 2011 находятся в Bishkek, поэтому слабый результат региона (выручка 27 050 против 47 280 в Osh, 2 убытка против 0) может объясняться ими, а не общей слабостью региона.
- Средняя выручка (8 855) зависит от выбора правила: при исключении по правилу B она падает на 20 %, поэтому рядом со средним нужно показывать медиану.

**3. Необходимые проверки**
- Для 2007 и 2011: кто и на каком основании утвердил скидку; сверка со справочником цен и себестоимости (ошибка ввода, например 50 % вместо 5 %); были ли акции и распродажи.
- Для 2002 и 2012: подтвердить, что цены относятся к дорогим штучным товарам, а не являются опечаткой.
- Проверить валюту и единицы измерения, полноту себестоимости (доставка, возвраты).

**4. Какие данные нужны для окончательного решения**
- дата заказа, клиент, менеджер, утвердивший скидку;
- категория товара и версия прайс-листа;
- условия акций и политика максимальной скидки;
- история тех же товаров и клиентов в других заказах;
- более длинная выборка: 12 заказов недостаточно для статистических выводов.

**Рекомендация.** Направить 2007 и 2011 на ручную проверку; до её завершения показывать показатели с этими заказами и без них; ввести контроль «скидка не выше безубыточной» при оформлении заказа.

## Дополнительное задание. Два правила выявления подозрительных заказов

In [9]:
# Правило 1 (строгое): очень высокая скидка и рентабельность ниже 5%
strict = (discount >= 0.30) & (margin < 0.05)
# Правило 2 (широкое): скидка >= 20% ИЛИ дорогой штучный товар ИЛИ рентабельность < 10%
broad = (discount >= 0.20) | ((unit_price >= 15_000) & (quantity <= 1)) | (margin < 0.10)

print("Строгое правило :", ids(strict), "| срабатываний:", strict.sum())
print("Широкое правило :", ids(broad), "| срабатываний:", broad.sum())
print("Пересечение     :", ids(strict & broad))
print("Только широкое  :", ids(broad & ~strict))

# учебная оценка по эталону «реальный убыток» (profit < 0)
for name, m in [("строгое", strict), ("широкое", broad)]:
    tp = (m & loss_mask).sum()
    print(f"{name}: найдено убыточных {tp} из {loss_mask.sum()}; точность={tp / m.sum():.0%}; полнота={tp / loss_mask.sum():.0%}")

Строгое правило : [2007 2011] | срабатываний: 2
Широкое правило : [2002 2004 2006 2007 2011 2012] | срабатываний: 6
Пересечение     : [2007 2011]
Только широкое  : [2002 2004 2006 2012]
строгое: найдено убыточных 2 из 2; точность=100%; полнота=100%
широкое: найдено убыточных 2 из 2; точность=33%; полнота=100%


**Вывод.** Строгое правило находит ровно два убыточных заказа (2007 и 2011), без ложных срабатываний, но «видит» только заранее описанный случай: убыток при скидке 15 % оно пропустит. Широкое правило тоже ловит оба убыточных заказа (полнота 100 %), но помечает 6 заказов из 12, добавляя 2002, 2004, 2006, 2012 (точность 33 %): половина выборки под подозрением - слишком много ложных срабатываний.

- **Для первичной проверки** подходит строгое правило: короткий список с высокой точностью, который можно сразу отдавать на ручную проверку.
- **Широкое правило** полезно как второй, «сторожевой» уровень на большой выборке (ранние признаки), но как основное оно перегружает проверяющих.

Оценка по эталону «реальный убыток» условна: на практике подозрительный заказ необязательно убыточен.